# Build a PD Model to Learn Terminology — Step by Step
### Shubham Rai | We will learn every term by USING it

**What is PD Model?**
PD = Probability of Default = Chance that a borrower will NOT pay back loan.
Example: PD = 0.8 means 80% chance this person will default. PD = 0.05 means 5% chance — safe.

We will build it on 1000 fake borrowers — same as your 100k real borrowers, but small so you can see.

## STEP 1: Create Data — What is a Borrower, Feature, Target?

**Borrower** = Person who took loan
**Feature** = Information about borrower (Income, Credit Score, DPD) — Input
**Target / Label** = Answer we want to predict (Default = 1 if he defaulted, 0 if he paid) — Output

We have 5 features + 1 target.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Create 1000 fake borrowers — like your real 100k data
n = 1000
data = pd.DataFrame({
    'income': np.random.randint(20000, 100000, n),  # Monthly income
    'credit_score': np.random.randint(500, 850, n), # CIBIL score
    'emi_to_income': np.random.uniform(0.1, 0.7, n), # EMI / Income — Feature you created
    'dpd': np.random.randint(0, 90, n), # Days Past Due — how many days late
    'loan_amount': np.random.randint(50000, 500000, n)
})

# TARGET: Default logic — if emi_to_income high AND credit_score low AND dpd high -> default
data['is_default'] = ((data['emi_to_income'] > 0.5) & (data['credit_score'] < 650) | (data['dpd'] > 60)).astype(int)

print(data.head(10))
print(f"\nTotal borrowers: {len(data)}")
print(f"Default rate: {data['is_default'].mean():.1%} — This is called BASE RATE")
print(f"Features (Inputs): income, credit_score, emi_to_income, dpd, loan_amount")
print(f"Target (Output to predict): is_default (0=Good, 1=Defaulter)")


## STEP 2: Train-Test Split — What is Training vs Testing?

**Training Data** = Data we use to TEACH model (70%) — like textbook
**Testing Data** = Data we HIDE from model to CHECK if it learned (30%) — like exam

If we test on same data we trained on, model will cheat by memorizing. So we split.

In [ ]:
from sklearn.model_selection import train_test_split

X = data[['income','credit_score','emi_to_income','dpd','loan_amount']] # Features = X
y = data['is_default'] # Target = y

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training: {len(X_train)} borrowers — Model will LEARN from these")
print(f"Testing: {len(X_test)} borrowers — Model will be TESTED on these, never seen before")
print(f"Train default rate: {y_train.mean():.1%} | Test default rate: {y_test.mean():.1%} — Should be similar")


## STEP 3: Train Model — What is Model, Logistic Regression?

**Model** = Formula that learns pattern: PD = f(income, credit_score, ...)
**Logistic Regression** = Simplest model for Yes/No (Default Yes/No). It gives probability 0 to 1.

**Fitting / Training** = Model looks at training data and finds weights.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# SCALING — Why? Income is 20000-100000, emi_to_income is 0.1-0.7. Without scaling, big numbers dominate.
# Scaling = Make all features same scale: (Value - Mean) / Std
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Scaling done — Now all features have Mean=0, Std=1')

model = LogisticRegression()
model.fit(X_train_scaled, y_train) # TRAINING — Model is LEARNING now
print('Model trained!')
print(f'Weights learned (Coefficients): {model.coef_}')
print('Positive weight = Increases PD, Negative weight = Decreases PD')


## STEP 4: Predict — What is Inference / Prediction?

**Inference** = Using trained model to predict for NEW borrower.
**Probability** = Model output 0 to 1. Example: 0.85 = 85% chance of default.
**Threshold** = Cutoff to decide Good vs Defaulter. Default 0.5 means if PD > 0.5, call him Defaulter.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Predict probability for test data
y_proba = model.predict_proba(X_test_scaled)[:,1] # Probability of Default
y_pred = (y_proba > 0.5).astype(int) # Threshold 0.5

print('First 5 borrowers:')
for i in range(5):
    print(f"Borrower {i+1}: True={y_test.iloc[i]}, Predicted PD={y_proba[i]:.2f}, Predicted Label={y_pred[i]}")

print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print('Confusion Matrix = TP (caught defaulters), FP (wrongly flagged good), FN (missed defaulters), TN (correct good)')
print(classification_report(y_test, y_pred, target_names=['Good','Defaulter']))


## STEP 5: Evaluation — What is Accuracy, Precision, Recall, KS, Gini?

**Accuracy** = Correct / Total. MISLEADING when default rate is low.
**Precision** = Out of predicted defaulters, how many actually defaulted? (If we say 100 defaulters, 70 actually default = 70% precision)
**Recall** = Out of actual defaulters, how many we caught? (If 100 real defaulters, we caught 71 = 71% recall) — MOST IMPORTANT for bank.
**ROC-AUC** = Ranking ability. 0.5 = Random, 1.0 = Perfect. 0.81 = Good.
**KS** = Max difference between Good and Bad distribution. >0.4 = Good.
**Gini** = 2*AUC - 1. 0.62 = Good.

In [ ]:
# Calculate KS, Gini, AUC
fpr, tpr, _ = roc_curve(y_test, y_proba)
ks = max(tpr - fpr)
auc = roc_auc_score(y_test, y_proba)
gini = 2*auc - 1
print(f'ROC-AUC: {auc:.3f} — Ranking ability')
print(f'KS: {ks:.3f} — Separation between Good and Bad')
print(f'Gini: {gini:.3f} — 2*AUC - 1')
print(f'Accuracy: {(y_pred==y_test).mean():.3f} — Misleading if imbalance')

# Try different threshold — See Recall improves
print('\n--- Threshold Tuning ---')
for thresh in [0.5, 0.3, 0.2]:
    y_pred_t = (y_proba > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    recall = tp/(tp+fn)
    print(f'Threshold {thresh}: Recall (caught defaulters) = {recall:.1%}, Flagged as high-risk = {(y_pred_t==1).mean():.1%}')


## STEP 6: XGBoost + SHAP — Why XGBoost is better?

**XGBoost** = Advanced model that learns non-linear patterns. Logistic is straight line, XGBoost is many trees.
**SHAP** = Explains WHY model said high risk for a specific borrower. Example: DTI contributed +0.2 to risk.

In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_proba_xgb = xgb_model.predict_proba(X_test)[:,1]
auc_xgb = roc_auc_score(y_test, y_proba_xgb)
print(f'XGBoost AUC: {auc_xgb:.3f} vs Logistic AUC: {auc:.3f}')
print('XGBoost usually better because it captures interactions like High DTI + High DPD = Very Risky')

# SHAP - Uncomment to see plot
# import shap
# explainer = shap.TreeExplainer(xgb_model)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test)


## YOUR GLOSSARY — Keep this

| Term | Simple Meaning |
|---|---|
| **PD** | Probability of Default — chance borrower will not pay |
| **Feature** | Input info about borrower (income, DPD) |
| **Target** | Output we predict (is_default) |
| **Training** | Teaching model |
| **Inference** | Using model to predict new borrower |
| **MSE / MAE** | How wrong model is — Mean Squared / Absolute Error |
| **Gradient** | Slope of hill — tells direction to reduce error |
| **Learning Rate** | How big step to take downhill |
| **Threshold** | Cutoff to call Good vs Defaulter |
| **Recall** | % of real defaulters we caught |
| **KS / Gini / AUC** | How well model separates Good vs Bad |
| **SHAP** | Why model said high risk |
| **PSI** | Has production data shifted? |

**Next:** Run this notebook cell by cell. Tell me which term still confuses you, and I will explain only that term with an example.